# SEG Regularization Tuning — QM8 Multi-Task Regression

Minimal SEG benchmark for multi-task regression (16 electronic excitation properties).

In [1]:
# === Setup ===
import sys
import random
from pathlib import Path
workspace_root = Path.cwd().parent
if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

import numpy as np
import pandas as pd
import torch

def set_seed(seed: int) -> None:
    """Set all random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print(f"Workspace: {workspace_root}")

Workspace: c:\Users\robsc\Home\Dev\molfusion2


In [2]:
# === Configuration ===
# TUNE THESE for regularization experiments

CONFIG = {
    # === REPRODUCIBILITY ===
    "seed": 42,                   # Global seed for reproducibility (None to disable)
    
    # Architecture
    "hidden_channels": 128,
    "K": 4,
    "num_layers": 3,
    "pool": "sum",
    "set2set_processing_steps": 6,
    
    # Fusion
    "fusion": "cross_mha",
    "fusion_dim": 64,
    "fusion_n_heads": 8,
    "text_projection_dim": 64,
    
    # === TEXT PROJECTION INITIALIZATION ===
    "text_proj_init": "xavier",      # "xavier" (scaled, recommended) or "kaiming" (PyTorch default)
    "text_proj_init_gain": 0.1,      # Gain for Xavier init (lower = smaller gradients)
    "freeze_text_proj": False,       # Freeze text projection (implicit regularization for small datasets)
    
    # === REGULARIZATION ===
    "dropout": 0.3,
    "fusion_dropout": 0.3,
    "head_dropout": 0.6,
    "weight_decay": 1e-1,
    
    # === HEAD ===
    "head_type": "mlp",
    "head_hidden_dim": 32,
    
    # Training
    "learning_rate": 1e-3,
    "batch_size": 64,
    "num_epochs": 100,
    "patience": 15,
    
    # === LR SCHEDULER ===
    "scheduler": "cosine",
    "scheduler_patience": 5,
    "scheduler_factor": 0.5,
    "min_lr": 1e-6,
    
    # === GRADIENT CLIPPING ===
    "grad_clip": None,            # Max gradient norm (None to disable). Recommended: 1.0-5.0
    
    # Multi-task regression
    "num_tasks": 16,
}

# Set global seed for reproducibility
if CONFIG["seed"] is not None:
    set_seed(CONFIG["seed"])
    print(f"=== Seed: {CONFIG['seed']} (reproducibility enabled) ===")
else:
    print("=== Seed: None (non-deterministic) ===")

print("=== QM8 Multi-Task Configuration ===")
print(f"Tasks: {CONFIG['num_tasks']} electronic properties")
print(f"Dropout: encoder={CONFIG['dropout']}, fusion={CONFIG['fusion_dropout']}, head={CONFIG['head_dropout']}")
print(f"Weight decay: {CONFIG['weight_decay']}")
print(f"Head: {CONFIG['head_type']} (hidden={CONFIG['head_hidden_dim']})")
print(f"LR: {CONFIG['learning_rate']}, batch: {CONFIG['batch_size']}")
print(f"Scheduler: {CONFIG['scheduler']} (patience={CONFIG['scheduler_patience']}, factor={CONFIG['scheduler_factor']})")
print(f"Pool: {CONFIG['pool']}" + (f" (steps={CONFIG['set2set_processing_steps']})" if CONFIG['pool'] == 'set2set' else ""))
print(f"Gradient clipping: {CONFIG['grad_clip']}")
print(f"Text proj init: {CONFIG['text_proj_init']} (gain={CONFIG['text_proj_init_gain']})")
print(f"Freeze text proj: {CONFIG['freeze_text_proj']}")

=== Seed: 42 (reproducibility enabled) ===
=== QM8 Multi-Task Configuration ===
Tasks: 16 electronic properties
Dropout: encoder=0.3, fusion=0.3, head=0.6
Weight decay: 0.1
Head: mlp (hidden=32)
LR: 0.001, batch: 64
Scheduler: cosine (patience=5, factor=0.5)
Pool: sum
Gradient clipping: None
Text proj init: xavier (gain=0.1)
Freeze text proj: False


In [3]:
# === Load QM8 Dataset ===
import deepchem as dc

SPLIT_TYPE = "random"  # Options: "scaffold" (paper uses this) or "random"

# Load QM8 with NormalizationTransformer (better training convergence)
tasks, datasets, transformers = dc.molnet.load_qm8(
    featurizer='ECFP',
    splitter='scaffold' if SPLIT_TYPE == 'scaffold' else 'random',
)
train_dc, valid_dc, test_dc = datasets
QM8_TASKS = tasks  # 16 electronic properties

# Extract normalized data (for training)
train_smiles = list(train_dc.ids)
train_y = train_dc.y.astype(np.float32)
valid_smiles = list(valid_dc.ids)
valid_y = valid_dc.y.astype(np.float32)
test_smiles = list(test_dc.ids)
test_y = test_dc.y.astype(np.float32)

# Store original (un-normalized) labels for evaluation
train_y_orig = train_y.copy()
valid_y_orig = valid_y.copy()
test_y_orig = test_y.copy()
for transformer in reversed(transformers):
    train_y_orig = transformer.untransform(train_y_orig)
    valid_y_orig = transformer.untransform(valid_y_orig)
    test_y_orig = transformer.untransform(test_y_orig)
train_y_orig = train_y_orig.astype(np.float32)
valid_y_orig = valid_y_orig.astype(np.float32)
test_y_orig = test_y_orig.astype(np.float32)

print(f"Split: {SPLIT_TYPE} | Train: {len(train_smiles)} | Valid: {len(valid_smiles)} | Test: {len(test_smiles)}")
print(f"Train labels shape: {train_y.shape}")
print(f"Transformers: {[type(t).__name__ for t in transformers]}")
print(f"Training on NORMALIZED data; will inverse-transform predictions for MAE evaluation")

No normalization for SPS. Feature removed!
No normalization for AvgIpc. Feature removed!
No normalization for NumAmideBonds. Feature removed!
No normalization for NumAtomStereoCenters. Feature removed!
No normalization for NumBridgeheadAtoms. Feature removed!
No normalization for NumHeterocycles. Feature removed!
No normalization for NumSpiroAtoms. Feature removed!
No normalization for NumUnspecifiedAtomStereoCenters. Feature removed!
No normalization for Phi. Feature removed!
Skipped loading some Tensorflow models, missing a dependency. No module named 'tensorflow'
Skipped loading modules with pytorch-geometric dependency, missing a dependency. No module named 'torch_geometric'
Skipped loading modules with transformers dependency. No module named 'transformers'
cannot import name 'HuggingFaceModel' from 'deepchem.models.torch_models' (c:\Users\robsc\Home\Dev\molfusion2\.venv\Lib\site-packages\deepchem\models\torch_models\__init__.py)
Skipped loading modules with pytorch-geometric depe

Split: random | Train: 17397 | Valid: 2175 | Test: 2175
Train labels shape: (17397, 16)
Transformers: ['NormalizationTransformer']
Training on NORMALIZED data; will inverse-transform predictions for MAE evaluation


In [4]:
# === Load Text Embeddings (from cache) ===
from utils.embedding_cache import EfficientEmbeddingCache

COT_TEXT_DIR = workspace_root / "cache" / "cot_texts"
COT_EMB_DIR  = workspace_root / "cache" / "cot_embeddings"

TASK = "quantum_fast"  # Cache prefix — matches {task}_text_embeddings_compact.npz

# Load compact npz cache (run the conversion cell below first if only .pkl exists)
npz_path = COT_EMB_DIR / f"{TASK}_text_embeddings_compact.npz"
cache = EfficientEmbeddingCache.load(npz_path)

all_smiles = train_smiles + valid_smiles + test_smiles
all_emb = cache.get_batch(all_smiles)               # (N, 3072) numpy float32
all_emb_t = torch.from_numpy(all_emb)                # → torch tensor

n_train = len(train_smiles)
n_valid = len(valid_smiles)
train_text_emb = all_emb_t[:n_train]
valid_text_emb = all_emb_t[n_train:n_train + n_valid]
test_text_emb  = all_emb_t[n_train + n_valid:]

print(f"✓ Loaded from {npz_path.name} ({len(cache)} molecules)")
print(f"  Embeddings: train={train_text_emb.shape}, valid={valid_text_emb.shape}, test={test_text_emb.shape}")

Loading embeddings from quantum_fast_text_embeddings_compact.npz...
  Loaded 21722 entries, dim=3072
  Memory mode: mapped
✓ Loaded from quantum_fast_text_embeddings_compact.npz (21722 molecules)
  Embeddings: train=torch.Size([17397, 3072]), valid=torch.Size([2175, 3072]), test=torch.Size([2175, 3072])


In [5]:
# === Initialize SEG for Multi-Task Regression ===
from models import SEGPredictor, SEGPredictorConfig

seg_config = SEGPredictorConfig(
    task="regression",
    num_tasks=CONFIG["num_tasks"],  # Multi-task: 16 outputs
    hidden_channels=CONFIG["hidden_channels"],
    K=CONFIG["K"],
    num_layers=CONFIG["num_layers"],
    dropout=CONFIG["dropout"],
    pool=CONFIG["pool"],
    set2set_processing_steps=CONFIG["set2set_processing_steps"],
    text_embedding_dim=3072,
    text_projection_dim=CONFIG["text_projection_dim"],
    text_proj_init=CONFIG["text_proj_init"],
    text_proj_init_gain=CONFIG["text_proj_init_gain"],
    freeze_text_proj=CONFIG["freeze_text_proj"],
    fusion=CONFIG["fusion"],
    fusion_dim=CONFIG["fusion_dim"],
    fusion_n_heads=CONFIG["fusion_n_heads"],
    fusion_dropout=CONFIG["fusion_dropout"],
    head_type=CONFIG["head_type"],
    head_hidden_dim=CONFIG["head_hidden_dim"],
    head_dropout=CONFIG["head_dropout"],
)

seg = SEGPredictor(config=seg_config)
print(f"SEG initialized: {CONFIG['num_tasks']} tasks, {CONFIG['fusion']} fusion, task=regression")
print(f"Text proj: init={CONFIG['text_proj_init']}, frozen={CONFIG['freeze_text_proj']}")

SEG initialized: 16 tasks, cross_mha fusion, task=regression
Text proj: init=xavier, frozen=False


In [6]:
# === Train SEG ===
print(f"Training SEG on QM8 ({CONFIG['num_tasks']} tasks, seed={CONFIG['seed']})\n")

history = seg.fit(
    smiles_list=train_smiles,
    labels=train_y,  # Shape: (N, 16) — normalized
    val_smiles=valid_smiles,
    val_labels=valid_y,
    text_embeddings=train_text_emb,
    val_text_embeddings=valid_text_emb,
    num_epochs=CONFIG["num_epochs"],
    batch_size=CONFIG["batch_size"],
    learning_rate=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
    patience=CONFIG["patience"],
    scheduler=CONFIG["scheduler"],
    scheduler_patience=CONFIG["scheduler_patience"],
    scheduler_factor=CONFIG["scheduler_factor"],
    min_lr=CONFIG["min_lr"],
    grad_clip=CONFIG["grad_clip"],
    seed=CONFIG["seed"] or 0,
    verbose=True,
)

# Count SEG parameters
n_params_seg = sum(p.numel() for p in seg._encoder.parameters())
n_params_seg += sum(p.numel() for p in seg._fusion.parameters())
n_params_seg += sum(p.numel() for p in seg._head.parameters())
if seg._text_proj: n_params_seg += sum(p.numel() for p in seg._text_proj.parameters())
print(f"\nSEG: {n_params_seg:,} parameters")

Training SEG on QM8 (16 tasks, seed=42)

Pre-computing molecular graphs for 17397 molecules...


[13:31:38] WARNING: not removing hydrogen atom without neighbors
[13:31:38] WARNING: not removing hydrogen atom without neighbors
[13:31:39] WARNING: not removing hydrogen atom without neighbors
[13:31:39] WARNING: not removing hydrogen atom without neighbors
[13:31:39] WARNING: not removing hydrogen atom without neighbors
[13:31:39] WARNING: not removing hydrogen atom without neighbors
[13:31:39] WARNING: not removing hydrogen atom without neighbors
[13:31:39] WARNING: not removing hydrogen atom without neighbors
[13:31:39] WARNING: not removing hydrogen atom without neighbors
[13:31:39] WARNING: not removing hydrogen atom without neighbors
[13:31:39] WARNING: not removing hydrogen atom without neighbors
[13:31:39] WARNING: not removing hydrogen atom without neighbors
[13:31:39] WARNING: not removing hydrogen atom without neighbors
[13:31:39] WARNING: not removing hydrogen atom without neighbors
[13:31:39] WARNING: not removing hydrogen atom without neighbors
[13:31:39] WARNING: not r

Pre-computed 17397/17397 valid graphs
Pre-computing molecular graphs for 2175 molecules...


[13:31:40] WARNING: not removing hydrogen atom without neighbors
[13:31:40] WARNING: not removing hydrogen atom without neighbors


Pre-computed 2175/2175 valid graphs
Training SEGPredictor on 17397 molecules...
Task: regression
Validation set: 2175 molecules
Fusion method: cross_mha
LR scheduler: cosine
Multi-label mode: 16 tasks
Epoch 001 | Train Loss: 0.9680 | Val RMSE: 0.9682 | LR: 1.00e-03
Epoch 005 | Train Loss: 0.9261 | Val RMSE: 0.9576 | LR: 9.94e-04
Epoch 010 | Train Loss: 0.9001 | Val RMSE: 0.9409 | LR: 9.76e-04
Epoch 015 | Train Loss: 0.8840 | Val RMSE: 0.9350 | LR: 9.46e-04
Epoch 020 | Train Loss: 0.8657 | Val RMSE: 0.9277 | LR: 9.05e-04
Epoch 025 | Train Loss: 0.8494 | Val RMSE: 0.9237 | LR: 8.54e-04
Epoch 030 | Train Loss: 0.8363 | Val RMSE: 0.9149 | LR: 7.94e-04
Epoch 035 | Train Loss: 0.8197 | Val RMSE: 0.9044 | LR: 7.27e-04
Epoch 040 | Train Loss: 0.8006 | Val RMSE: 0.8976 | LR: 6.55e-04
Epoch 045 | Train Loss: 0.7962 | Val RMSE: 0.8936 | LR: 5.79e-04
Epoch 050 | Train Loss: 0.7789 | Val RMSE: 0.8878 | LR: 5.01e-04
Epoch 055 | Train Loss: 0.7654 | Val RMSE: 0.8857 | LR: 4.22e-04
Epoch 060 | Train L

In [7]:
# === Test Set Evaluation (Per-Task MAE) ===

# Get predictions (in normalized scale) and inverse-transform to original Hartree scale
test_preds_norm = seg.predict_batch(test_smiles, text_embeddings=test_text_emb)
valid_preds_norm = seg.predict_batch(valid_smiles, text_embeddings=valid_text_emb)

test_preds = test_preds_norm.copy()
valid_preds = valid_preds_norm.copy()
for transformer in reversed(transformers):
    test_preds = transformer.untransform(test_preds)
    valid_preds = transformer.untransform(valid_preds)
test_preds = test_preds.astype(np.float32)
valid_preds = valid_preds.astype(np.float32)

# Per-task MAE on original scale
def compute_mae(y_true, y_pred):
    mask = ~np.isnan(y_true) & ~np.isnan(y_pred)
    return np.mean(np.abs(y_true[mask] - y_pred[mask])) if mask.sum() > 0 else np.nan

print("=== Per-Task MAE (×1e-3) — Original Scale ===")
print(f"{'Task':40} {'Valid MAE':>15} {'Test MAE':>15}")
print("-" * 72)

test_maes, valid_maes = [], []
for i, task in enumerate(QM8_TASKS):
    valid_mae = compute_mae(valid_y_orig[:, i], valid_preds[:, i])
    test_mae = compute_mae(test_y_orig[:, i], test_preds[:, i])
    valid_maes.append(valid_mae)
    test_maes.append(test_mae)
    print(f"{task:40} {valid_mae * 1000:>15.4f} {test_mae * 1000:>15.4f}")

mean_valid_mae = np.nanmean(valid_maes)
mean_test_mae = np.nanmean(test_maes)
print("-" * 72)
print(f"{'Mean':40} {mean_valid_mae * 1000:>15.4f} {mean_test_mae * 1000:>15.4f}")
print(f"{'Std':40} {np.nanstd(valid_maes) * 1000:>15.4f} {np.nanstd(test_maes) * 1000:>15.4f}")
print()
print(f"Config: dropout={CONFIG['dropout']}, wd={CONFIG['weight_decay']}, fusion={CONFIG['fusion']}, pool={CONFIG['pool']}")

[13:37:16] WARNING: not removing hydrogen atom without neighbors
[13:37:16] WARNING: not removing hydrogen atom without neighbors
[13:37:16] WARNING: not removing hydrogen atom without neighbors
[13:37:16] WARNING: not removing hydrogen atom without neighbors
[13:37:17] WARNING: not removing hydrogen atom without neighbors
[13:37:17] WARNING: not removing hydrogen atom without neighbors


=== Per-Task MAE (×1e-3) — Original Scale ===
Task                                           Valid MAE        Test MAE
------------------------------------------------------------------------
E1-CC2                                           29.2521         30.0534
E2-CC2                                           21.9394         22.5631
f1-CC2                                           32.2501         31.2992
f2-CC2                                           52.9469         53.9803
E1-PBE0                                          31.3352         32.3782
E2-PBE0                                          24.9049         25.4239
f1-PBE0                                          30.6800         30.5383
f2-PBE0                                          42.1837         41.6173
E1-PBE0                                          31.3352         32.3782
E2-PBE0                                          24.9049         25.4239
f1-PBE0                                          30.6800         30.5383
f2-PB